# `tab:agreement_kappa` — 4-row regeneration

Recomputes every cell of `tab:agreement_kappa` in `report/latex/acl_latex.tex` from raw sources, so the table has a single source of truth.

Column changes vs the current LaTeX:
- drop `% agree`
- keep `Cohen's κ` on binary harm labels
- add `Cosine sim` on intent text (SBERT `all-MiniLM-L6-v2`, same encoder used in `compute_semantic_similarity`)

Rows:
1. **Human – Human** — pairwise within duplicate-prompt groups in `Jazhyc/wildguard-annotated-intents`.
2. **Human – WildGuardMix** — `Annotator Harm` (binarised) vs `Dataset Harm` on the full annotated set.
3. **Human – GPT-OSS-120B** — `true_harm_binary` / `true_intent` vs `predicted_harm` / `generated_intent` from the vanilla_generation run on n=1,724.
4. **GPT-OSS-120B – WildGuardMix** — model's `predicted_harm` vs `Dataset Harm`, joined by `ID`.

Parse-failure policy: rows where the model failed to emit a parseable `predicted_harm` are coerced to `safe` (the negative class), so rows 3 and 4 keep the full n=1,724 instead of dropping ~20 rows. Empty `generated_intent` is treated as the empty string for cosine, applying the same penalty principle. Cosine is N/A for rows 2 and 4 (WildGuard has no intent text).

In [1]:
import json
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import concatenate_datasets, load_dataset
from sklearn.metrics import cohen_kappa_score

from intention_jailbreak.model_generation.evaluate_generations import compute_semantic_similarity

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "CLAUDE.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

GPT_OSS_JSONL = REPO_ROOT / "data" / "safety_experiment" / "annotated-intents-full" / "openai_gpt-oss-120b_vanilla_generation.jsonl"
assert GPT_OSS_JSONL.exists(), GPT_OSS_JSONL

HARMFUL_5LVL = {"Completely Harmful", "Uncertain Harmful"}
SAFE_5LVL = {"Completely Safe", "Uncertain Safe"}

def to_binary_annotator(label):
    if label in HARMFUL_5LVL:
        return "harmful"
    if label in SAFE_5LVL:
        return "safe"
    return None

In [2]:
ds = load_dataset("Jazhyc/wildguard-annotated-intents")
annot = concatenate_datasets([ds["train"], ds["validation"], ds["test"]]).to_pandas()
annot = annot.dropna(subset=["Annotator Harm"]).copy()
annot["Annotator Harm Binary"] = annot["Annotator Harm"].map(to_binary_annotator)
annot["Dataset Harm Binary"] = annot["Dataset Harm"].str.lower()

with GPT_OSS_JSONL.open() as f:
    gpt = pd.DataFrame(json.loads(line) for line in f)

print(f"annotated rows (non-null Annotator Harm): {len(annot)}")
print(f"gpt-oss rows                            : {len(gpt)}")
print(f"annot binary distrib: {annot['Annotator Harm Binary'].value_counts(dropna=False).to_dict()}")
print(f"dataset (WG) distrib: {annot['Dataset Harm Binary'].value_counts(dropna=False).to_dict()}")
print(f"gpt-oss parse fails — predicted_harm None: {gpt['predicted_harm'].isna().sum()}")
print(f"gpt-oss parse fails — generated_intent empty: {gpt['generated_intent'].fillna('').str.strip().eq('').sum()}")

annotated rows (non-null Annotator Harm): 1724
gpt-oss rows                            : 1724
annot binary distrib: {'harmful': 871, 'safe': 853}
dataset (WG) distrib: {'safe': 903, 'harmful': 821}
gpt-oss parse fails — predicted_harm None: 20
gpt-oss parse fails — generated_intent empty: 21


## Row 1 — Human – Human

Duplicate-prompt groups are reconstructed by grouping on the `Prompt` text (per CLAUDE.md: `Duplicate ID` only tags 2nd+ occurrences, so grouping on it misses 1st-occurrence rows). For each group, enumerate all unordered pairs of annotators; pool the `(rater_a, rater_b)` tuples and compute one Cohen's κ on the pool. Same pairs are used for the cosine column.

In [3]:
groups = [g for _, g in annot.groupby("Prompt") if len(g) >= 2]

harm_a, harm_b, int_a, int_b = [], [], [], []
for g in groups:
    rows = g.to_dict("records")
    for r1, r2 in combinations(rows, 2):
        if r1["Annotator Harm Binary"] is None or r2["Annotator Harm Binary"] is None:
            continue
        harm_a.append(r1["Annotator Harm Binary"])
        harm_b.append(r2["Annotator Harm Binary"])
        int_a.append((r1.get("Intent") or "").strip())
        int_b.append((r2.get("Intent") or "").strip())

hh_kappa = cohen_kappa_score(harm_a, harm_b)
hh_agree = float(np.mean([a == b for a, b in zip(harm_a, harm_b)]))
hh_cos = compute_semantic_similarity(int_a, int_b)
hh_cos_mean = float(np.mean(hh_cos))

print(f"groups (prompts with ≥2 annotations): {len(groups)}")
print(f"harm/intent pairs       : {len(harm_a)}")
print(f"Cohen's κ (harm)        : {hh_kappa:.4f}")
print(f"Raw agreement (harm)    : {hh_agree:.4f}")
print(f"Mean cosine sim (intent): {hh_cos_mean:.4f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


groups (prompts with ≥2 annotations): 225
harm/intent pairs       : 945
Cohen's κ (harm)        : 0.5481
Raw agreement (harm)    : 0.7767
Mean cosine sim (intent): 0.6162


## Row 2 — Human – WildGuardMix

All 1,724 rows. `Annotator Harm` is collapsed 5→2; `Dataset Harm` is already binary (`Safe` / `Harmful`).

In [4]:
hw = annot.dropna(subset=["Annotator Harm Binary", "Dataset Harm Binary"]).copy()
hw_kappa = cohen_kappa_score(hw["Annotator Harm Binary"], hw["Dataset Harm Binary"])
hw_agree = float((hw["Annotator Harm Binary"] == hw["Dataset Harm Binary"]).mean())

print(f"n: {len(hw)}")
print(f"Cohen's κ    : {hw_kappa:.4f}")
print(f"Raw agreement: {hw_agree:.4f}")
print()
print(pd.crosstab(hw["Annotator Harm Binary"], hw["Dataset Harm Binary"], margins=True))

n: 1724
Cohen's κ    : 0.4504
Raw agreement: 0.7251

Dataset Harm Binary    harmful  safe   All
Annotator Harm Binary                     
harmful                    609   262   871
safe                       212   641   853
All                        821   903  1724


## Row 3 — Human – GPT-OSS-120B

All n=1,724 rows from the vanilla_generation JSONL. Parse failures: `predicted_harm` = `None` → `safe`; empty `generated_intent` → empty string fed to SBERT (yields a low cosine, applying the same failure-penalty principle as the κ side).

In [5]:
hg = gpt.copy()
hg["true_harm_binary"] = hg["true_harm_binary"].str.lower()
hg["predicted_harm"] = hg["predicted_harm"].fillna("safe").str.lower()
hg_kappa = cohen_kappa_score(hg["true_harm_binary"], hg["predicted_harm"])
hg_agree = float((hg["true_harm_binary"] == hg["predicted_harm"]).mean())

true_intent = gpt["true_intent"].fillna("").astype(str).tolist()
gen_intent = gpt["generated_intent"].fillna("").astype(str).tolist()
hg_cos = compute_semantic_similarity(true_intent, gen_intent)
hg_cos_mean = float(np.mean(hg_cos))

print(f"n: {len(hg)}")
print(f"Cohen's κ (harm)        : {hg_kappa:.4f}")
print(f"Raw agreement (harm)    : {hg_agree:.4f}")
print(f"Mean cosine sim (intent): {hg_cos_mean:.4f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


n: 1724
Cohen's κ (harm)        : 0.5042
Raw agreement (harm)    : 0.7523
Mean cosine sim (intent): 0.5815


## Row 4 — GPT-OSS-120B – WildGuardMix

Join model predictions on the annotated dataset by `ID`, compare `predicted_harm` (with `None` → `safe`) to `Dataset Harm`. Cosine N/A — WildGuard has no intent text.

In [6]:
merged = gpt[["id", "predicted_harm"]].merge(
    annot[["ID", "Dataset Harm Binary"]].rename(columns={"ID": "id"}),
    on="id",
    how="inner",
).dropna(subset=["Dataset Harm Binary"]).copy()
merged["predicted_harm"] = merged["predicted_harm"].fillna("safe").str.lower()

gw_kappa = cohen_kappa_score(merged["predicted_harm"], merged["Dataset Harm Binary"])
gw_agree = float((merged["predicted_harm"] == merged["Dataset Harm Binary"]).mean())

print(f"n: {len(merged)}")
print(f"Cohen's κ    : {gw_kappa:.4f}")
print(f"Raw agreement: {gw_agree:.4f}")
print()
print(pd.crosstab(merged["predicted_harm"], merged["Dataset Harm Binary"], margins=True))

n: 1724
Cohen's κ    : 0.5020
Raw agreement: 0.7500



Dataset Harm Binary  harmful  safe   All
predicted_harm                          
harmful                  663   273   936
safe                     158   630   788
All                      821   903  1724


## Summary — drop-in values for `tab:agreement_kappa`

Sorted by Cohen's κ descending.

In [7]:
summary = pd.DataFrame(
    [
        {"Comparison": "Human – Human",                 "n": len(harm_a),  "Cohen's κ": hh_kappa, "Cosine sim": hh_cos_mean},
        {"Comparison": "Human – WildGuardMix",          "n": len(hw),      "Cohen's κ": hw_kappa, "Cosine sim": np.nan},
        {"Comparison": "Human – GPT-OSS-120B",          "n": len(hg),      "Cohen's κ": hg_kappa, "Cosine sim": hg_cos_mean},
        {"Comparison": "GPT-OSS-120B – WildGuardMix",   "n": len(merged),  "Cohen's κ": gw_kappa, "Cosine sim": np.nan},
    ]
).sort_values("Cohen's κ", ascending=False).reset_index(drop=True)
summary.style.format({"Cohen's κ": "{:.4f}", "Cosine sim": "{:.4f}"})

,Comparison,n,Cohen's κ,Cosine sim
0,Human – Human,945,0.5481,0.6162
1,Human – GPT-OSS-120B,1724,0.5042,0.5815
2,GPT-OSS-120B – WildGuardMix,1724,0.5020,nan
3,Human – WildGuardMix,1724,0.4504,nan
